# Notebook 3: Sparse Index + Crash Recovery (the BEST practices)

Notebook 2 gave us segments and cheap retention, but two real-world problems remain:

1. **Lookup is slow.** To find record #12345 we scan from the start of a segment.
2. **Crashes leave half-written records.** If the process dies mid-write, the tail of the active segment is garbage, and a naive reader will explode.

We fix both here.

## Setup

```bash
cd 02-distributed-primitives/segmented-log
uv sync
```

Select the `.venv` kernel in VS Code. If it doesn't appear, reload the window (`Cmd+Shift+P` -> **Reload Window**).

## Part 1: Sparse index

Kafka keeps an `.index` file next to each `.log` segment. It maps **some** logical offsets (every Nth record) to a byte position in the segment. This is a *sparse* index: small enough to fit in RAM, but good enough that a lookup does one seek + a short linear scan.

**Analogy:** a book has a table of contents (sparse - chapter granularity), not an index of every word. Finding a sentence = jump to chapter, then read a few pages.

```
offset 0   -> byte 0
offset 16  -> byte 240
offset 32  -> byte 512   <- to find offset 37, seek here, then read 5 records.
offset 48  -> byte 764
```

In [1]:
import os, tempfile, glob

class IndexedSegmentedLog:
    """Segmented log + in-memory sparse index for O(log N) offset lookup."""

    INDEX_EVERY = 16  # record one entry every 16 records (tunable)

    def __init__(self, dir_, max_segment_bytes=1024):
        self.dir = dir_
        os.makedirs(dir_, exist_ok=True)
        self.max_bytes = max_segment_bytes
        self.active_id = 0
        self.next_offset = 0           # global logical offset across all segments
        # index[segment_id] = list of (logical_offset, byte_position)
        self.index = {0: [(0, 0)]}
        self.seg_base = {0: 0}         # first logical offset of each segment
        self._open_active()

    def _seg_path(self, sid):
        return os.path.join(self.dir, f'segment-{sid:08d}.log')

    def _open_active(self):
        self.fp = open(self._seg_path(self.active_id), 'ab')
        self.active_size = self.fp.tell()

    def append(self, record: bytes):
        framed = len(record).to_bytes(4, 'big') + record
        if self.active_size + len(framed) > self.max_bytes and self.active_size > 0:
            self._roll()
        pos_in_seg = self.active_size
        self.fp.write(framed)
        self.active_size += len(framed)
        # Sparse index: only every Nth record goes in.
        if self.next_offset % self.INDEX_EVERY == 0:
            self.index.setdefault(self.active_id, []).append(
                (self.next_offset, pos_in_seg))
        self.next_offset += 1

    def _roll(self):
        self.fp.close()
        self.active_id += 1
        self.seg_base[self.active_id] = self.next_offset
        self.index[self.active_id] = [(self.next_offset, 0)]
        self._open_active()

    def read_at(self, offset: int) -> bytes:
        """Return the record at logical offset. Uses the sparse index."""
        if offset < 0 or offset >= self.next_offset:
            raise IndexError(offset)
        # 1. Find which segment holds this offset (the last segment whose base <= offset).
        sid = max(s for s, base in self.seg_base.items() if base <= offset)
        # 2. In that segment's index, find the largest entry with logical_offset <= offset.
        entries = self.index[sid]
        lo, hi = 0, len(entries) - 1
        while lo < hi:
            mid = (lo + hi + 1) // 2
            if entries[mid][0] <= offset:
                lo = mid
            else:
                hi = mid - 1
        start_offset, start_pos = entries[lo]
        # 3. Seek there and scan forward (offset - start_offset) records.
        self.fp.flush()
        with open(self._seg_path(sid), 'rb') as f:
            f.seek(start_pos)
            skip = offset - start_offset
            for _ in range(skip):
                n = int.from_bytes(f.read(4), 'big')
                f.seek(n, 1)  # relative seek, skip the payload
            n = int.from_bytes(f.read(4), 'big')
            return f.read(n)

    def close(self):
        self.fp.close()

WORKDIR = tempfile.mkdtemp(prefix='seglog_idx_')
log = IndexedSegmentedLog(WORKDIR, max_segment_bytes=1024)
for i in range(500):
    log.append(f'event-{i:05d}'.encode())

print('segments written:', log.active_id + 1)
print('index entries per segment:', {sid: len(e) for sid, e in log.index.items()})

segments written: 8
index entries per segment: {0: 6, 1: 5, 2: 5, 3: 5, 4: 6, 5: 5, 6: 5, 7: 3}


### Lookups are now near-instant

No matter where the record is in the log, we do at most ONE seek + a short scan of `INDEX_EVERY - 1` records. Compare this with "open segment, read from byte 0 until you counted N records" - which is O(N).

In [2]:
import time

for target in [0, 7, 42, 123, 499]:
    t0 = time.perf_counter()
    rec = log.read_at(target)
    dt_us = (time.perf_counter() - t0) * 1e6
    print(f'offset {target:4d} -> {rec!r:20s}  ({dt_us:6.1f} us)')

assert log.read_at(250) == b'event-00250'
print('OK: sparse index lookup works')

offset    0 -> b'event-00000'        ( 398.5 us)
offset    7 -> b'event-00007'        ( 137.5 us)
offset   42 -> b'event-00042'        (  58.4 us)
offset  123 -> b'event-00123'        ( 227.3 us)
offset  499 -> b'event-00499'        (3988.0 us)
OK: sparse index lookup works


## Part 2: Crash recovery with a torn write

Appends are not atomic. A power loss or kill -9 in the middle of a write can leave:

- a length header but no payload, or
- a truncated payload.

A naive reader will try to read `length` bytes, hit EOF, and crash - or worse, silently return garbage. **Best practice:** on startup, scan the active segment and truncate back to the last *complete* record.

> Production systems (Kafka, PostgreSQL) also store a **CRC checksum** per record so they > can detect silent bit-rot, not just truncation. We'll keep it simple and just handle > truncation here.

In [3]:
import os, tempfile

# 1. Write a clean log of 5 records.
crash_dir = tempfile.mkdtemp(prefix='crash_')
seg = os.path.join(crash_dir, 'segment-00000000.log')
with open(seg, 'ab') as f:
    for i in range(5):
        r = f'msg-{i}'.encode()
        f.write(len(r).to_bytes(4, 'big'))
        f.write(r)
clean_size = os.path.getsize(seg)
print('clean segment size:', clean_size)

# 2. Simulate a crash mid-write: append a length header plus ONLY 2 of the 10 payload bytes.
with open(seg, 'ab') as f:
    f.write((10).to_bytes(4, 'big'))
    f.write(b'XX')
print('corrupted size:', os.path.getsize(seg))

clean segment size: 45
corrupted size: 51


In [4]:
def recover(path: str) -> int:
    """Scan a segment, return the byte offset of the end of the last VALID record.

    Anything past that point is a torn write and should be truncated."""
    good_end = 0
    with open(path, 'rb') as f:
        while True:
            hdr = f.read(4)
            if len(hdr) < 4:
                break  # no room for another header -> stop
            n = int.from_bytes(hdr, 'big')
            payload = f.read(n)
            if len(payload) < n:
                break  # torn write: payload shorter than length said
            good_end = f.tell()
    return good_end

good = recover(seg)
print(f'last valid byte: {good} (file size was {os.path.getsize(seg)})')

# Truncate the corruption away. This is the SAFE recovery step.
with open(seg, 'r+b') as f:
    f.truncate(good)
print('after truncate, size:', os.path.getsize(seg))
assert os.path.getsize(seg) == clean_size
print('OK: segment recovered to last intact record')

last valid byte: 45 (file size was 51)
after truncate, size: 45
OK: segment recovered to last intact record


## Recap: the progression

| Step | What changed | Why it matters |
|------|--------------|----------------|
| NB1  | One giant file | Simple, but deletes rewrite everything (O(n)) |
| NB2  | Split into segments + roll | Retention becomes `unlink` (O(1) per segment) |
| NB3  | Sparse index + recovery | Fast random access + survives crashes |

## Best practices you now know

- **Roll by size OR time**, whichever trips first. This caps segment size *and* freshness.
- **Zero-pad segment IDs** (`segment-00000042.log`) so directory listing order == log order.
- Keep an **in-memory sparse index**; persist a snapshot of it next to each closed segment so you don't have to rebuild on startup.
- Add a **CRC per record** to detect silent corruption, not just torn writes.
- On startup, **truncate the active segment** back to the last valid record.
- Only the **active** segment is mutable; older segments are immutable -> safe to share, memory-map, compress, or ship to S3.

## Further reading

- [Kafka: a distributed messaging system for log processing](https://notes.stephenholiday.com/Kafka.pdf) - the paper.
- [Kafka docs - log segments & index files](https://kafka.apache.org/documentation/#log).
- [PostgreSQL WAL internals](https://www.postgresql.org/docs/current/wal-internals.html).
- [Designing Data-Intensive Applications, Chapter 3](https://dataintensive.net/) - log-structured storage.
